In [1]:
from openai import OpenAI
import os
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv("OPENROUTER_API_KEY"),
)


In [2]:
db = {} 

In [3]:
import tqdm, os
import random
import time
reviews = os.listdir("../human_reviews")
def build_embedding(i): 
    while True:
        try:
            if reviews[i] in db.keys():
                print(f"Embedding for {reviews[i]} already exists, skipping...")
                return db[reviews[i]] 
            else:
                with open(f"../human_reviews/{reviews[i]}", "r") as f:
                    content = f.read()

                embedding = client.embeddings.create(
                    model="google/gemini-embedding-001",
                    input=content,
                    encoding_format="float"
                )
                db[reviews[i]] = embedding.data[0].embedding
            return embedding.data[0].embedding
        except Exception as e:
            print(f"Error embedding {reviews[i]}: {e}. Retrying...")
            time.sleep(random.uniform(1, 10))


In [4]:
db.keys()

dict_keys([])

In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed

with ThreadPoolExecutor(max_workers=50) as executor:
    futures = {executor.submit(build_embedding, i): i for i in range(len(reviews))}
    results = {}
    for f in tqdm.tqdm(as_completed(futures), total=len(futures)):
        idx = futures[f]
        results[idx] = f.result() 

100%|██████████| 7763/7763 [03:58<00:00, 32.50it/s]


In [6]:
keys = list(db.keys())
values = list(db.values())

In [8]:
import numpy as np

values = np.array(values)

In [110]:
query_embedding = client.embeddings.create( 
    model="google/gemini-embedding-001",
    input="having sex",
    encoding_format="float" 
)

In [111]:
keys[(np.array(query_embedding.data[0].embedding) @ values.T).argmax()]

'ZAyuwJYN8N.md'

In [114]:
with open(f"./human_reviews_embeddings.pkl", "wb") as f:
    import pickle
    pickle.dump(db, f)

In [112]:
filename = keys[(np.array(query_embedding.data[0].embedding) @ values.T).argsort()[-1]]
with open(f"../human_reviews/{filename}", "r") as f:
    print(f"\nMost relevant review for query:\n{f.read()}")


Most relevant review for query:
# InterMask: 3D Human Interaction Generation via Collaborative Masked Modeling

- Decision: Accept (Poster)
- Scores: 6, 6, 6

## Abstract
Generating realistic 3D human-human interactions from textual descriptions remains a challenging task. Existing approaches, typically based on diffusion models, often produce results lacking realism and fidelity. In this work, we introduce *InterMask*, a novel framework for generating human interactions using collaborative masked modeling in discrete space. InterMask first employs a VQ-VAE to transform each motion sequence into a 2D discrete motion token map. Unlike traditional 1D VQ token maps, it better preserves fine-grained spatio-temporal details and promotes *spatial awareness* within each token. Building on this representation, InterMask utilizes a generative masked modeling framework to collaboratively model the tokens of two interacting individuals. This is achieved by employing a transformer architecture sp